# Experimento adicional: autorreflexões com orçamento alto

Este notebook parte do experimento completo `91ccab5e5028`. Ele reutiliza as respostas das questões de treino, os pares top-1 e os resultados das condições baseline/externas já produzidos com as reflexões do GPT-5.4 Petrobras. Somente as autorreflexões de Phi-2, DeepSeek-R1-Distill-Llama-8B e Llama 3.1 8B — e suas duas condições de validação — são geradas novamente.

Diferenças desta variante:

- as autorreflexões recebem um teto muito maior;
- não há segunda tentativa: a geração já começa com o teto alto;
- uma reflexão encerrada por `length` é preservada e utilizada;
- se o DeepSeek truncar dentro de `<think>`, o conteúdo parcial é preservado como reflexão experimental, em vez de virar texto vazio;
- quando a reflexão completa não cabe no contexto do Phi-2 durante a validação, o começo e o fim são preservados e o recorte é auditado; a reflexão completa nunca é apagada do checkpoint.

Neste notebook, o termo informal *ollama* foi interpretado como o mesmo modelo `llama3.1-8b`, executado pelo backend vLLM.

In [ ]:
# Execute esta célula antes de qualquer import de torch/vLLM.
import os

GPU = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
print(f"CUDA_VISIBLE_DEVICES={GPU}")
print("Se torch ou vLLM já tiverem sido importados neste kernel, reinicie o kernel agora.")

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path
from typing import Any
import hashlib
import json
import math
import re

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

import rmcq.backends.base as backend_base
from rmcq.backends import GenParams, get_backend
from rmcq.prompts import (
    REFLECTION_DEPTHS,
    STUDENT_REFLECTION_PROMPTS,
    build_answer_prompt,
    build_reflection_prompt,
    build_transfer_prompt,
)
from run_experiment import (
    PHI2_STOP_SEQUENCES,
    cache_key,
    cached_generate,
    load_jsonl,
    resolve_answers,
    save_csv,
    save_json,
    save_jsonl,
    summarize,
    validation_answer_budget,
)

plt.style.use("seaborn-v0_8-whitegrid")

BASE_EXPERIMENT_ID = "91ccab5e5028"
VARIANT_NAME = "high_budget_keep_truncated_v1"
MODELS = ["phi2", "deepseek-r1-distill-llama-8b", "llama3.1-8b"]
DATASETS = ["aqua", "arc", "logiqa2", "openbookqa"]
JUDGE_MODEL = "llama3.1-8b"
BACKEND_KIND = "vllm"
REFLECTION_TEMPERATURE = 0.7
BATCH_SIZE = 256
N_SIMILARITY_BINS = 10
FRESH_REFLECTIONS = False
FRESH_VALIDATION = False

# Phi-2 é limitado por sua janela nativa de 2.048 tokens. Os demais recebem 4.096.
REFLECTION_CEILINGS = {
    "phi2": {"simple": 1024, "complex": 1024},
    "deepseek-r1-distill-llama-8b": {"simple": 4096, "complex": 4096},
    "llama3.1-8b": {"simple": 4096, "complex": 4096},
}

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "run_experiment.py").exists() and (candidate / "rmcq").is_dir():
            return candidate
    raise FileNotFoundError("Não encontrei a raiz do repositório Reflection-MCQ.")

ROOT = find_repo_root()
EXCHANGE = ROOT / "experiment_exchange" / BASE_EXPERIMENT_ID
RESULT_DIR = ROOT / "data" / "results" / "reflection_high_budget" / BASE_EXPERIMENT_ID
WORK_DIR = RESULT_DIR / "work"
ANALYSIS_DIR = RESULT_DIR / "analysis"
PLOTS_DIR = ANALYSIS_DIR / "plots"
for directory in (WORK_DIR, ANALYSIS_DIR, PLOTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Base:      {EXCHANGE}")
print(f"Variante:  {RESULT_DIR}")

In [ ]:
manifest_path = EXCHANGE / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifesto não encontrado: {manifest_path}")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest.get("models") != MODELS:
    raise ValueError(f"Modelos da base são {manifest.get('models')}, não {MODELS}.")
if manifest.get("datasets") != DATASETS:
    raise ValueError(f"Datasets da base são {manifest.get('datasets')}, não {DATASETS}.")

pairs = [
    row
    for dataset in DATASETS
    for row in load_jsonl(EXCHANGE / "pairs" / f"{dataset}.jsonl")
]
source_items = {row["source_uid"]: row["source_item"] for row in pairs}
student_rows = {
    model: {
        row["source_uid"]: row
        for row in load_jsonl(EXCHANGE / "students" / model / "train.jsonl")
    }
    for model in MODELS
}
teacher_rows = {
    model: {
        row["source_uid"]: row
        for row in load_jsonl(EXCHANGE / "teacher" / "student_reflections" / f"{model}.jsonl")
    }
    for model in MODELS
}

print(f"Pares de validação: {len(pairs):,}")
print(f"Fontes de treino únicas: {len(source_items):,}")
for model in MODELS:
    resolved = sum(row.get("correct") is not None for row in student_rows[model].values())
    print(f"{model}: {resolved:,}/{len(source_items):,} respostas de treino resolvidas")

## Geração das novas autorreflexões

O backend normal remove blocos `<think>` e transforma um bloco aberto truncado em texto vazio. Para esta variante, a geração abaixo captura o texto bruto retornado pelo vLLM. Quando existir uma resposta visível depois de `</think>`, ela continua sendo preferida. Quando não existir, o conteúdo parcial é mantido e marcado como `partial_think_kept`.

In [ ]:
canonical_strip_thinking = backend_base.strip_thinking
THINK_TAG_RE = re.compile(r"</?think(?:\s+[^>]*)?>", re.IGNORECASE)

@contextmanager
def preserve_raw_generation_text():
    original = backend_base.strip_thinking
    backend_base.strip_thinking = lambda text: (text or "").strip()
    try:
        yield
    finally:
        backend_base.strip_thinking = original

def usable_reflection(raw_text: str) -> tuple[str, str]:
    raw_text = (raw_text or "").strip()
    visible = canonical_strip_thinking(raw_text)
    if visible:
        return visible, "visible_output"
    fallback = THINK_TAG_RE.sub("", raw_text).strip()
    if fallback:
        return fallback, "partial_think_kept"
    return "", "empty"

def prompt_digest(model: str, depth: str, prompt: str, output_budget: int) -> str:
    payload = json.dumps(
        {
            "variant": VARIANT_NAME,
            "model": model,
            "depth": depth,
            "temperature": REFLECTION_TEMPERATURE,
            "output_budget": output_budget,
            "prompt": prompt,
        },
        sort_keys=True, ensure_ascii=False,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]

def high_budget_reflections(
    backend: Any,
    model: str,
    depth: str,
    prompts: dict[str, str],
    cache_path: Path,
    ceiling: int,
) -> dict[str, dict[str, Any]]:
    cached = (
        {} if FRESH_REFLECTIONS or not cache_path.exists()
        else {row["key"]: row for row in load_jsonl(cache_path)}
    )
    scheduled: dict[int, list[tuple[str, str, str, int]]] = defaultdict(list)

    for uid, prompt in prompts.items():
        prompt_tokens = len(backend.render_token_ids(backend.tokenizer, prompt))
        capacity = int(backend.max_len) - prompt_tokens
        if capacity <= 0:
            cached[uid] = {
                "key": uid, "prompt_hash": None, "raw_text": "",
                "reflection": "", "finish_reason": "prompt_context_exhausted",
                "prompt_tokens": prompt_tokens, "completion_tokens": 0,
                "output_budget": 0, "extraction": "empty",
                "kept_despite_length": False,
            }
            continue
        effective = min(ceiling, capacity)
        # Agrupar em múltiplos de 64 evita centenas de chamadas pequenas ao vLLM.
        output_budget = effective if effective < 64 else max(64, (effective // 64) * 64)
        digest = prompt_digest(model, depth, prompt, output_budget)
        current = cached.get(uid)
        if current and current.get("prompt_hash") == digest:
            continue
        scheduled[output_budget].append((uid, prompt, digest, prompt_tokens))

    total_missing = sum(len(group) for group in scheduled.values())
    print(f"{model} {depth}: total={len(prompts)} cache={len(prompts)-total_missing} missing={total_missing}")

    for output_budget, entries in sorted(scheduled.items(), reverse=True):
        for start in range(0, len(entries), BATCH_SIZE):
            batch = entries[start:start + BATCH_SIZE]
            stop = PHI2_STOP_SEQUENCES if model == "phi2" else ()
            with preserve_raw_generation_text():
                generations = backend.generate(
                    [prompt for _uid, prompt, _digest, _tokens in batch],
                    GenParams(
                        max_new_tokens=output_budget,
                        temperature=REFLECTION_TEMPERATURE,
                        stop=stop,
                    ),
                    desc=f"{model} self reflection {depth}",
                )
            for (uid, _prompt, digest, prompt_tokens), generation in zip(batch, generations):
                raw_text = generation.text
                reflection, extraction = usable_reflection(raw_text)
                cached[uid] = {
                    "key": uid,
                    "prompt_hash": digest,
                    "raw_text": raw_text,
                    "reflection": reflection,
                    "finish_reason": generation.finish_reason,
                    "prompt_tokens": generation.prompt_tokens or prompt_tokens,
                    "completion_tokens": generation.completion_tokens,
                    "output_budget": output_budget,
                    "extraction": extraction,
                    "kept_despite_length": generation.finish_reason == "length" and bool(reflection),
                }
            save_jsonl(cache_path, cached.values())

    save_jsonl(cache_path, cached.values())
    return {uid: cached[uid] for uid in prompts}


In [ ]:
new_reflections: dict[str, dict[str, dict[str, dict[str, Any]]]] = {}

for model in MODELS:
    new_reflections[model] = {}
    eligible = {
        uid: row for uid, row in student_rows[model].items()
        if row.get("correct") is not None and (row.get("response") or "").strip()
    }
    with get_backend(model, kind=BACKEND_KIND) as backend:
        for depth in REFLECTION_DEPTHS:
            prompts = {
                uid: build_reflection_prompt(
                    source_items[uid], row["response"], bool(row["correct"]),
                    depth=depth, perspective="student",
                )
                for uid, row in eligible.items()
            }
            new_reflections[model][depth] = high_budget_reflections(
                backend=backend, model=model, depth=depth, prompts=prompts,
                cache_path=WORK_DIR / "reflections" / model / f"{depth}.jsonl",
                ceiling=REFLECTION_CEILINGS[model][depth],
            )

save_json(RESULT_DIR / "protocol.json", {
    "variant": VARIANT_NAME,
    "base_experiment_id": BASE_EXPERIMENT_ID,
    "models": MODELS,
    "datasets": DATASETS,
    "judge_model": JUDGE_MODEL,
    "backend": BACKEND_KIND,
    "reflection_temperature": REFLECTION_TEMPERATURE,
    "reflection_ceilings": REFLECTION_CEILINGS,
    "keep_length_outputs": True,
    "keep_partial_think_outputs": True,
    "student_reflection_prompts": STUDENT_REFLECTION_PROMPTS,
    "generated_validation_conditions": ["self_simple", "self_complex"],
    "reused_validation_conditions": ["baseline", "teacher_simple", "teacher_complex"],
})
print("Autorreflexões concluídas.")

In [ ]:
reflection_audit_rows = []
for model in MODELS:
    for depth in REFLECTION_DEPTHS:
        rows = list(new_reflections[model][depth].values())
        reflection_audit_rows.append({
            "model": model,
            "depth": depth,
            "n": len(rows),
            "usable": sum(bool(row.get("reflection")) for row in rows),
            "ended_by_length": sum(row.get("finish_reason") == "length" for row in rows),
            "length_outputs_kept": sum(bool(row.get("kept_despite_length")) for row in rows),
            "partial_think_kept": sum(row.get("extraction") == "partial_think_kept" for row in rows),
            "mean_completion_tokens": (
                sum(row.get("completion_tokens", 0) for row in rows) / len(rows) if rows else 0
            ),
        })
reflection_audit = pd.DataFrame(reflection_audit_rows)
reflection_audit.to_csv(ANALYSIS_DIR / "reflection_generation_audit.csv", index=False)
reflection_audit

## Validação com as novas reflexões

Somente `self_simple` e `self_complex` são geradas novamente. Baseline, reflexão externa simples e reflexão externa complexa são copiadas dos resultados finais do experimento-base, pois nada nessas três condições mudou. Isso reduz 60% das gerações de validação sem alterar a comparação experimental.

Quando necessário, somente a reflexão é recortada para caber na janela. O enunciado recuperado e a questão de validação nunca são truncados. O recorte mantém metade dos tokens do começo e metade do fim, porque a lição costuma aparecer no final.

In [ ]:
GENERATED_CONDITIONS = ["self_simple", "self_complex"]
REUSED_CONDITIONS = ["baseline", "teacher_simple", "teacher_complex"]
ALL_CONDITIONS = ["baseline", "self_simple", "self_complex", "teacher_simple", "teacher_complex"]
BASE_OUTCOMES_PATH = (
    ROOT / "data" / "results" / "reflection_top1" / BASE_EXPERIMENT_ID
    / "analysis" / "all_outcomes.jsonl"
)

def decode_head_tail(tokenizer: Any, token_ids: list[int], keep_tokens: int) -> str:
    if keep_tokens >= len(token_ids):
        return tokenizer.decode(token_ids, skip_special_tokens=True).strip()
    head_n = (keep_tokens + 1) // 2
    tail_n = keep_tokens // 2
    head = tokenizer.decode(token_ids[:head_n], skip_special_tokens=True).strip()
    tail = tokenizer.decode(token_ids[-tail_n:], skip_special_tokens=True).strip() if tail_n else ""
    return f"{head}\n[... reflection shortened only to fit context ...]\n{tail}".strip()

def fit_transfer_prompt(
    backend: Any, val_item: dict[str, Any], source_item: dict[str, Any],
    source_response: str, source_correct: bool, reflection: str,
    answer_reserve: int,
) -> tuple[str | None, dict[str, Any]]:
    full_prompt = build_transfer_prompt(
        val_item, source_item, source_response, source_correct, reflection
    )
    full_prompt_tokens = len(backend.render_token_ids(backend.tokenizer, full_prompt))
    full_reflection_tokens = backend.count_tokens(reflection)
    if full_prompt_tokens + answer_reserve <= backend.max_len:
        return full_prompt, {
            "reflection_tokens_full": full_reflection_tokens,
            "reflection_tokens_used": full_reflection_tokens,
            "reflection_clipped_for_context": False,
            "prompt_tokens": full_prompt_tokens,
        }

    reflection_ids = backend.tokenizer(
        reflection, add_special_tokens=False
    )["input_ids"]
    if hasattr(reflection_ids, "tolist"):
        reflection_ids = reflection_ids.tolist()

    low, high = 1, len(reflection_ids)
    best: tuple[str, int, int] | None = None
    while low <= high:
        keep = (low + high) // 2
        shortened = decode_head_tail(backend.tokenizer, reflection_ids, keep)
        candidate = build_transfer_prompt(
            val_item, source_item, source_response, source_correct, shortened
        )
        candidate_tokens = len(backend.render_token_ids(backend.tokenizer, candidate))
        if candidate_tokens + answer_reserve <= backend.max_len:
            best = (candidate, keep, candidate_tokens)
            low = keep + 1
        else:
            high = keep - 1

    if best is None:
        return None, {
            "reflection_tokens_full": full_reflection_tokens,
            "reflection_tokens_used": 0,
            "reflection_clipped_for_context": True,
            "eval_method": "transfer_context_exceeded_even_after_reflection_clip",
        }

    candidate, used, candidate_tokens = best
    return candidate, {
        "reflection_tokens_full": full_reflection_tokens,
        "reflection_tokens_used": used,
        "reflection_clipped_for_context": True,
        "prompt_tokens": candidate_tokens,
    }

generated_by_model = {}
items_by_model = {}
metadata_by_model = {}
unavailable_by_model = {}
cache_dirs = {}

for model in MODELS:
    prompts, items, metadata = {}, {}, {}
    unavailable = []
    cache_dir = WORK_DIR / "validation" / model
    cache_dirs[model] = cache_dir

    with get_backend(model, kind=BACKEND_KIND) as backend:
        initial_answer_tokens = validation_answer_budget(model)
        retry_answer_tokens = initial_answer_tokens + max(128, initial_answer_tokens // 2)

        for pair in pairs:
            val_item = pair["validation_item"]
            source_item = pair["source_item"]
            source_uid = pair["source_uid"]
            attempt = student_rows[model][source_uid]

            for condition in GENERATED_CONDITIONS:
                key = cache_key(pair["dataset"], pair["val_uid"], condition)
                fit_meta = {
                    "reflection_tokens_full": None,
                    "reflection_tokens_used": None,
                    "reflection_clipped_for_context": False,
                }

                if condition == "baseline":
                    prompt = build_answer_prompt(val_item)
                    prompt_tokens = len(backend.render_token_ids(backend.tokenizer, prompt))
                    if prompt_tokens + retry_answer_tokens > backend.max_len:
                        unavailable.append({
                            "model": model, "dataset": pair["dataset"],
                            "val_uid": pair["val_uid"], "source_uid": source_uid,
                            "similarity": pair["similarity"], "condition": condition,
                            "response": "", "finish_reason": "not_generated",
                            "selected_answer": None, "correct": None,
                            "eval_method": "validation_context_exceeded",
                        })
                        continue
                else:
                    if attempt.get("correct") is None or not (attempt.get("response") or "").strip():
                        unavailable.append({
                            "model": model, "dataset": pair["dataset"],
                            "val_uid": pair["val_uid"], "source_uid": source_uid,
                            "similarity": pair["similarity"], "condition": condition,
                            "response": "", "finish_reason": "not_generated",
                            "selected_answer": None, "correct": None,
                            "eval_method": "source_answer_unavailable",
                        })
                        continue

                    author, depth = condition.split("_", 1)
                    if author == "self":
                        reflection_row = new_reflections[model][depth].get(source_uid, {})
                        reflection = reflection_row.get("reflection")
                        reflection_origin = reflection_row.get("extraction")
                    else:
                        reflection_row = teacher_rows[model].get(source_uid, {})
                        reflection = (reflection_row.get("reflections") or {}).get(depth)
                        reflection_origin = "gpt_5_4_petrobras"

                    if not (reflection or "").strip():
                        unavailable.append({
                            "model": model, "dataset": pair["dataset"],
                            "val_uid": pair["val_uid"], "source_uid": source_uid,
                            "similarity": pair["similarity"], "condition": condition,
                            "response": "", "finish_reason": "not_generated",
                            "selected_answer": None, "correct": None,
                            "eval_method": "source_reflection_unavailable",
                        })
                        continue

                    prompt, fit_meta = fit_transfer_prompt(
                        backend, val_item, source_item, attempt["response"],
                        bool(attempt["correct"]), reflection, retry_answer_tokens,
                    )
                    fit_meta["reflection_origin"] = reflection_origin
                    if prompt is None:
                        unavailable.append({
                            "model": model, "dataset": pair["dataset"],
                            "val_uid": pair["val_uid"], "source_uid": source_uid,
                            "similarity": pair["similarity"], "condition": condition,
                            "response": "", "finish_reason": "not_generated",
                            "selected_answer": None, "correct": None,
                            **fit_meta,
                        })
                        continue

                prompts[key] = prompt
                items[key] = val_item
                metadata[key] = {"pair": pair, "condition": condition, **fit_meta}

        generated_by_model[model] = cached_generate(
            backend=backend, path=cache_dir / "validation.jsonl", prompts=prompts,
            max_tokens=initial_answer_tokens, batch_size=BATCH_SIZE,
            fresh=FRESH_VALIDATION, description=f"{model} high-budget-reflection validation",
            stop=PHI2_STOP_SEQUENCES if model == "phi2" else (),
        )

    items_by_model[model] = items
    metadata_by_model[model] = metadata
    unavailable_by_model[model] = unavailable

print("Geração da validação concluída; iniciando julgamento de fallbacks.")

In [ ]:
verdicts_by_model = {}
with get_backend(JUDGE_MODEL, kind=BACKEND_KIND) as judge_backend:
    for model in MODELS:
        verdicts_by_model[model] = resolve_answers(
            judge_backend, cache_dirs[model], "validation",
            generated_by_model[model], items_by_model[model],
            BATCH_SIZE, FRESH_VALIDATION,
        )

if not BASE_OUTCOMES_PATH.exists():
    raise FileNotFoundError(
        f"Resultados-base não encontrados: {BASE_OUTCOMES_PATH}. "
        "Eles são necessários para reutilizar baseline e reflexões externas."
    )
base_outcomes = load_jsonl(BASE_OUTCOMES_PATH)
outcomes = [
    {**row, "result_source": "base_experiment"}
    for row in base_outcomes
    if row.get("model") in MODELS and row.get("condition") in REUSED_CONDITIONS
]
for model in MODELS:
    outcomes.extend(
        {**row, "result_source": VARIANT_NAME}
        for row in unavailable_by_model[model]
    )
    for key, meta in metadata_by_model[model].items():
        pair = meta["pair"]
        outcomes.append({
            "model": model,
            "dataset": pair["dataset"],
            "val_uid": pair["val_uid"],
            "source_uid": pair["source_uid"],
            "similarity": pair["similarity"],
            "condition": meta["condition"],
            "response": generated_by_model[model][key]["text"],
            "finish_reason": generated_by_model[model][key]["finish_reason"],
            "selected_answer": verdicts_by_model[model][key]["selected_answer"],
            "correct": verdicts_by_model[model][key]["correct"],
            "eval_method": verdicts_by_model[model][key]["eval_method"],
            "reflection_tokens_full": meta.get("reflection_tokens_full"),
            "reflection_tokens_used": meta.get("reflection_tokens_used"),
            "reflection_clipped_for_context": meta.get("reflection_clipped_for_context", False),
            "reflection_origin": meta.get("reflection_origin"),
            "result_source": VARIANT_NAME,
        })

save_jsonl(ANALYSIS_DIR / "all_outcomes.jsonl", outcomes)
accuracy_rows = summarize(outcomes)
save_csv(ANALYSIS_DIR / "accuracy.csv", accuracy_rows)
save_json(RESULT_DIR / "finish_receipt.json", {
    "variant": VARIANT_NAME,
    "base_experiment_id": BASE_EXPERIMENT_ID,
    "rows": len(outcomes),
    "generated_validation_rows": len(MODELS) * len(GENERATED_CONDITIONS) * len(pairs),
    "reused_validation_rows": len(MODELS) * len(REUSED_CONDITIONS) * len(pairs),
    "expected_rows": len(MODELS) * len(ALL_CONDITIONS) * len(pairs),
    "unresolved_conditions": sum(row.get("correct") is None for row in outcomes),
    "reflections_clipped_for_context": sum(
        bool(row.get("reflection_clipped_for_context")) for row in outcomes
    ),
    "complete": True,
})

print(f"Resultados: {ANALYSIS_DIR}")
print(f"Linhas: {len(outcomes):,} (esperado: {len(MODELS) * len(ALL_CONDITIONS) * len(pairs):,})")

## Auditoria e comparação agregada

A tabela abaixo mostra acurácia e cobertura. Os deltas comparam reflexão externa e autorreflexão de mesma profundidade.

In [ ]:
results = pd.DataFrame(outcomes)
results["similarity"] = pd.to_numeric(results["similarity"], errors="coerce")
results["correct_value"] = pd.to_numeric(results["correct"], errors="coerce")

coverage = (
    results.groupby(["model", "dataset", "condition"], as_index=False)
    .agg(n=("val_uid", "size"), resolved=("correct_value", "count"), accuracy=("correct_value", "mean"))
)
coverage["coverage"] = coverage["resolved"] / coverage["n"]
coverage.to_csv(ANALYSIS_DIR / "coverage.csv", index=False)

overall = (
    results.groupby(["model", "condition"], as_index=False)
    .agg(accuracy=("correct_value", "mean"), resolved=("correct_value", "count"))
)
pivot = overall.pivot(index="model", columns="condition", values="accuracy")
pivot["external_minus_self_simple"] = pivot["teacher_simple"] - pivot["self_simple"]
pivot["external_minus_self_complex"] = pivot["teacher_complex"] - pivot["self_complex"]
pivot.to_csv(ANALYSIS_DIR / "external_advantage.csv")
pivot.style.format("{:.1%}")

In [ ]:
CONDITION_ORDER = ["baseline", "self_simple", "self_complex", "teacher_simple", "teacher_complex"]
CONDITION_LABELS = {
    "baseline": "Baseline",
    "self_simple": "Self-ref. simples (teto alto)",
    "self_complex": "Self-ref. complexa (teto alto)",
    "teacher_simple": "Ref. externa simples (GPT-5.4)",
    "teacher_complex": "Ref. externa complexa (GPT-5.4)",
}
CONDITION_COLORS = {
    "baseline": "#222222", "self_simple": "#1f77b4",
    "self_complex": "#17becf", "teacher_simple": "#d62728",
    "teacher_complex": "#ff7f0e",
}
DATASET_ORDER = ["aqua", "arc", "logiqa2", "openbookqa"]
DATASET_LABELS = {"aqua": "AQuA", "arc": "ARC", "logiqa2": "LogiQA 2.0", "openbookqa": "OpenBookQA"}
MODEL_LABELS = {
    "phi2": "Phi-2",
    "deepseek-r1-distill-llama-8b": "DeepSeek-R1-Distill-Llama-8B",
    "llama3.1-8b": "Llama 3.1 8B",
}

pair_similarity = (
    results.loc[results["similarity"].notna(), ["dataset", "val_uid", "similarity"]]
    .drop_duplicates(["dataset", "val_uid"]).copy()
)

def assign_quantile_bins(group: pd.DataFrame) -> pd.DataFrame:
    group = group.copy()
    n_bins = min(N_SIMILARITY_BINS, group["similarity"].nunique(), len(group))
    group["similarity_bin"] = (
        0 if n_bins < 2
        else pd.qcut(group["similarity"], q=n_bins, labels=False, duplicates="drop").astype(int)
    )
    return group

pair_similarity = pd.concat(
    [assign_quantile_bins(group.drop(columns="dataset")).assign(dataset=dataset)
     for dataset, group in pair_similarity.groupby("dataset", sort=False)],
    ignore_index=True,
)
plot_rows = results.merge(
    pair_similarity[["dataset", "val_uid", "similarity_bin"]],
    on=["dataset", "val_uid"], how="inner", validate="many_to_one",
)
bin_centers = (
    pair_similarity.groupby(["dataset", "similarity_bin"], as_index=False)
    .agg(similarity=("similarity", "mean"))
)
binned_accuracy = (
    plot_rows.groupby(["model", "dataset", "condition", "similarity_bin"], as_index=False)
    .agg(accuracy=("correct_value", "mean"), n_resolved=("correct_value", "count"))
    .merge(bin_centers, on=["dataset", "similarity_bin"], how="left")
)
binned_accuracy.head()

In [ ]:
def model_slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.casefold()).strip("-")

def plot_model(model: str):
    model_data = binned_accuracy[binned_accuracy["model"] == model]
    fig, axes = plt.subplots(2, 2, figsize=(13, 9.2), sharey=True)
    handles = {}

    for ax, dataset in zip(axes.flat, DATASET_ORDER):
        subset = model_data[model_data["dataset"] == dataset]
        for condition in CONDITION_ORDER:
            line_data = subset[subset["condition"] == condition].sort_values("similarity")
            line_data = line_data[line_data["accuracy"].notna()]
            if line_data.empty:
                continue
            line, = ax.plot(
                line_data["similarity"], line_data["accuracy"],
                color=CONDITION_COLORS[condition], marker="o",
                markersize=4.5, linewidth=2, label=CONDITION_LABELS[condition],
            )
            handles[condition] = line
        ax.set_title(DATASET_LABELS[dataset])
        ax.set_xlabel("Similaridade média no quantil")
        ax.set_ylabel("Acurácia")
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.grid(alpha=0.25)

    ordered_handles = [handles[c] for c in CONDITION_ORDER if c in handles]
    ordered_labels = [CONDITION_LABELS[c] for c in CONDITION_ORDER if c in handles]
    fig.legend(ordered_handles, ordered_labels, loc="lower center", ncol=3,
               bbox_to_anchor=(0.5, 0.005), frameon=False)
    fig.suptitle(
        f"{MODEL_LABELS.get(model, model)}: reflexões próprias com teto alto vs. GPT-5.4",
        fontsize=15, y=0.995,
    )
    fig.tight_layout(rect=(0, 0.09, 1, 0.97))
    output = PLOTS_DIR / f"accuracy_by_similarity_{model_slug(model)}.png"
    fig.savefig(output, dpi=180, bbox_inches="tight")
    fig.savefig(output.with_suffix(".pdf"), bbox_inches="tight")
    print(f"Salvo: {output}")
    plt.show()
    return fig

figures = {model: plot_model(model) for model in MODELS}

## Arquivos produzidos

- `protocol.json`: configuração congelada da variante;
- `work/reflections/<model>/<depth>.jsonl`: texto bruto e reflexão utilizável, inclusive saídas em `length`;
- `analysis/reflection_generation_audit.csv`: quantidade de truncamentos preservados;
- `analysis/all_outcomes.jsonl`: todas as condições de validação;
- `analysis/accuracy.csv` e `coverage.csv`: métricas agregadas;
- `analysis/external_advantage.csv`: diferença entre reflexão externa e autorreflexão;
- `analysis/plots/`: grids em PNG e PDF.

Para retomar uma execução interrompida, mantenha `FRESH_REFLECTIONS=False` e `FRESH_VALIDATION=False` e execute novamente a partir da geração de autorreflexões.